# 04 · Interfaz — elige un órgano y genera

Menú desplegable + parámetros de muestreo. Debajo de cada imagen aparece la **probabilidad que el clasificador de órganos asigna al órgano pedido**, y al final las **métricas globales** guardadas por el notebook 03.

> Requiere haber corrido 02 (modelo) y 03 (clasificador y métricas). Las imágenes se guardan en `results/interfaz/`.
> Las imágenes sintéticas son **solo para investigación/educación, no para diagnóstico**.

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/fractal_histopatologia_256/code")   # carpeta con fg_*.py

!pip -q install "timm==0.9.12" scikit-learn pandas ipywidgets
import json, numpy as np, torch
import fg_config as cfg, fg_common as fg, fg_eval as fe
fg.mount_drive(); fg.setup_repo(); fg.gpu_info()
X, y, classes, tr_idx, va_idx = fg.load_clean_data()
model, _ = fg.load_for_inference("best_fp16.pth")
clf = fe.train_classifier(X, y, tr_idx, va_idx, classes)            # se carga de Drive
cg = f"{cfg.RES_DIR}/config_gen.json"
CFG0 = json.load(open(cg))["cfg"] if os.path.exists(cg) else cfg.CFG
mg = f"{cfg.RES_DIR}/metricas_globales.json"
METR = json.load(open(mg)) if os.path.exists(mg) else None

In [ ]:
import ipywidgets as W
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

dd = W.Dropdown(options=classes, description="Órgano:", layout=W.Layout(width="320px"))
sl_cfg = W.FloatSlider(value=CFG0, min=1, max=10, step=0.5, description="CFG:")
sl_temp = W.FloatSlider(value=1.0, min=0.8, max=1.2, step=0.01, description="Temp:")
sl_it = W.Dropdown(options=[("Rápido (32,8,8)", (32, 8, 8)), ("Calidad (64,16,16)", (64, 16, 16))],
                   value=(32, 8, 8), description="Pasos:")
n_img = W.IntSlider(value=2, min=1, max=4, description="Imágenes:")
seed = W.IntText(value=0, description="Semilla:")
btn = W.Button(description="Generar", button_style="primary", icon="play")
out = W.Output()
os.makedirs(f"{cfg.RES_DIR}/interfaz", exist_ok=True)

def on_click(_):
    with out:
        clear_output(wait=True)
        print("Generando (a 256 px puede tardar)...")
        o = dd.value; ci = classes.index(o)
        imgs = fg.generate_many(model, [ci] * n_img.value, bs=n_img.value, seed=seed.value,
                                cfg_scale=sl_cfg.value, temperature=sl_temp.value, num_iter_list=sl_it.value)
        p = fe.clf_probs(clf, imgs)[:, ci]
        clear_output(wait=True)
        fig, axs = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4.3))
        for k, a in enumerate(np.atleast_1d(axs)):
            a.imshow(imgs[k].permute(1, 2, 0).numpy()); a.axis("off")
            a.set_title(f"{o}\nP(órgano)={p[k]:.2f}", fontsize=10)
            fg.to_pil(imgs[k]).save(f"{cfg.RES_DIR}/interfaz/{o}_seed{seed.value}_{k}.png")
        plt.show()
        if METR:
            print(f"Métricas del modelo -> FID {METR['FID']:.1f} (piso {METR['FID_piso']:.1f}) | KID {METR['KID']:.4f} | "
                  f"IS {METR['IS']:.2f} | exactitud generadas {METR['exactitud_gen']:.2f} (reales {METR['exactitud_real']:.2f})")
        else:
            print("Corre el notebook 03 para ver FID / KID / IS aquí.")
btn.on_click(on_click)
display(W.VBox([dd, sl_cfg, sl_temp, sl_it, n_img, seed, btn, out]))